---
## Section 0: Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import polars as pl
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print("folium not available — skipping folium maps")

os.makedirs('outputs/eda', exist_ok=True)
os.makedirs('outputs/maps', exist_ok=True)

CITY_COLORS = {'nyc': '#FF5A5F', 'la': '#00A699', 'chi': '#FC642D'}
CITY_LABELS = {'nyc': 'NYC', 'la': 'Los Angeles', 'chi': 'Chicago'}
PLOTLY_TPL  = 'plotly_white'
DPI         = 150
SEED        = 42

def save_fig(fig, name):
    fig.savefig(f'outputs/eda/{name}.png', dpi=DPI, bbox_inches='tight')
    plt.close(fig)

def save_plotly(fig, name):
    fig.write_html(f'outputs/eda/{name}.html')

print("Libraries loaded.")

In [ ]:
df = pl.read_parquet('data/airbnb_cleaned.parquet')
print(f"Shape : {df.shape}")
print(f"\nSchema (first 15 cols):")
for col, dtype in list(zip(df.columns, df.dtypes))[:15]:
    print(f"  {col:<40} {dtype}")

GRID_DEG = 0.001
try:
    ws_cache = pl.read_parquet('data/walkscore_cache.parquet')
    ws_valid = (
        ws_cache
        .filter(pl.col('status') == 1)
        .select(['snapped_lat', 'snapped_lon', 'walkscore', 'transit_score', 'bike_score'])
    )
    df = (
        df
        .with_columns([
            ((pl.col('latitude')  / GRID_DEG).round(0) * GRID_DEG).alias('snapped_lat'),
            ((pl.col('longitude') / GRID_DEG).round(0) * GRID_DEG).alias('snapped_lon'),
        ])
        .join(ws_valid, on=['snapped_lat', 'snapped_lon'], how='left')
        .drop(['snapped_lat', 'snapped_lon'])
    )
    ws_hit = df['walkscore'].is_not_null().mean()
    print(f"Walk score merge hit rate : {ws_hit:.1%}")
except Exception as e:
    print(f"Walk score cache unavailable: {e}")

WALK_COLS = [c for c in ['walkscore', 'transit_score', 'bike_score'] if c in df.columns]
HAS_WALKSCORE = len(WALK_COLS) > 0 and df[WALK_COLS[0]].is_not_null().sum() > 100
print(f"Walk score columns available: {HAS_WALKSCORE}")

In [ ]:
print(f"Dataset shape : {df.shape}")

print("\nNull counts (non-amenity cols with nulls):")
null_counts = {
    col: df[col].is_null().sum()
    for col in df.columns
    if not col.startswith('amenity_') and df[col].is_null().sum() > 0
}
for col, n in sorted(null_counts.items(), key=lambda x: -x[1])[:15]:
    print(f"  {col:<45} {n}")

display_cols = ['city', 'room_type', 'price_usd', 'log_price', 'bedrooms', 'amenity_count', 'neighbourhood_cleansed']
print(f"\nFirst 3 rows:")
print(df.select(display_cols).head(3))

In [ ]:
city_stats = (
    df
    .group_by('city')
    .agg([
        pl.len().alias('Listings'),
        pl.col('price_usd').mean().round(2).alias('Mean $'),
        pl.col('price_usd').median().round(2).alias('Median $'),
        pl.col('price_usd').std().round(2).alias('Std $'),
    ])
    .with_columns(pl.col('city').replace(CITY_LABELS).alias('City'))
    .drop('city')
    .sort('Mean $', descending=True)
)
print("City-level price statistics:")
print(city_stats)

---
## Section 1: Price Distribution

### 1a & 1b. Raw vs log-price distribution

In [ ]:
price_arr    = df['price_usd'].to_numpy()
logprice_arr = df['log_price'].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
sns.histplot(price_arr, bins=80, kde=True, color='#FF5A5F', ax=ax)
skew_val = float(df['price_usd'].skew())
ax.set_title('Raw Price Distribution (USD)', fontsize=13, fontweight='bold')
ax.set_xlabel('price_usd'); ax.set_ylabel('Count')
ax.annotate(f'Skewness = {skew_val:.2f}', xy=(0.62, 0.85), xycoords='axes fraction', fontsize=11)

ax = axes[1]
sns.histplot(logprice_arr, bins=80, kde=True, color='#00A699', ax=ax)
log_skew = float(df['log_price'].skew())
ax.set_title('Log-Price Distribution (ln USD)', fontsize=13, fontweight='bold')
ax.set_xlabel('log_price'); ax.set_ylabel('Count')
ax.annotate(f'Skewness = {log_skew:.2f}', xy=(0.05, 0.85), xycoords='axes fraction', fontsize=11)

fig.suptitle('Price vs Log-Price: The Case for a Log Transform', fontsize=14, y=1.01)
plt.tight_layout()
save_fig(fig, '1ab_price_distribution')
plt.show()
print(f"Raw skewness: {skew_val:.2f}  |  Log skewness: {log_skew:.2f}")

### 1c. Price by city — KDE + box plot (Plotly)

In [ ]:
# KDE overlay by city — violin in horizontal orientation
fig_kde = go.Figure()
for city, label in CITY_LABELS.items():
    vals = df.filter(pl.col('city') == city)['log_price'].drop_nulls().to_numpy()
    fig_kde.add_trace(go.Violin(
        x=vals, name=label, side='positive', orientation='h',
        line_color=CITY_COLORS[city], fillcolor=CITY_COLORS[city],
        opacity=0.5, bandwidth=0.08, showlegend=True,
    ))
fig_kde.update_layout(
    template=PLOTLY_TPL, title='Log-Price KDE by City',
    xaxis_title='log_price (ln USD)', yaxis_visible=False, height=350,
)
save_plotly(fig_kde, '1c_logprice_kde_city')
fig_kde.show()

In [ ]:
# Box plot — price_usd by city, log scale
df_box = df.with_columns(pl.col('city').replace(CITY_LABELS).alias('City'))
fig_box = px.box(
    df_box, x='City', y='price_usd',
    color='City',
    color_discrete_map={v: CITY_COLORS[k] for k, v in CITY_LABELS.items()},
    log_y=True,
    title='Price Distribution by City (log scale)',
    labels={'price_usd': 'price_usd (log scale)', 'City': ''},
    template=PLOTLY_TPL,
)
save_plotly(fig_box, '1c_price_box_city')
fig_box.show()

### 1d. Price by room type — violin plot (Plotly)

In [ ]:
df_viol = df.with_columns(pl.col('city').replace(CITY_LABELS).alias('City'))
fig_viol = px.violin(
    df_viol, x='room_type', y='log_price',
    color='City',
    color_discrete_map={v: CITY_COLORS[k] for k, v in CITY_LABELS.items()},
    box=True, points=False,
    title='Log-Price by Room Type and City',
    labels={'room_type': 'Room Type', 'log_price': 'log_price (ln USD)'},
    template=PLOTLY_TPL,
    category_orders={'room_type': ['Entire home/apt', 'Private room', 'Hotel room', 'Shared room']},
)
save_plotly(fig_viol, '1d_logprice_roomtype_violin')
fig_viol.show()

### 1e. Median price vs bedrooms per city

In [ ]:
bed_med = (
    df
    .filter(pl.col('bedrooms').is_between(0, 6))
    .with_columns(pl.col('bedrooms').cast(pl.Int32))
    .group_by(['city', 'bedrooms'])
    .agg(pl.col('price_usd').median().alias('median_price'))
    .sort(['city', 'bedrooms'])
)

fig, ax = plt.subplots(figsize=(10, 5))
for city, label in CITY_LABELS.items():
    sub = bed_med.filter(pl.col('city') == city)
    ax.plot(sub['bedrooms'].to_numpy(), sub['median_price'].to_numpy(),
            marker='o', label=label, color=CITY_COLORS[city], linewidth=2.5, markersize=7)
ax.set_title('Median Price vs Number of Bedrooms (capped at 6)', fontsize=13, fontweight='bold')
ax.set_xlabel('Bedrooms'); ax.set_ylabel('Median Price (USD)')
ax.legend(title='City'); ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, '1e_price_vs_bedrooms')
plt.show()

### 1f. Amenity count vs log-price — scatter + regression line

In [ ]:
samp = df.select(['amenity_count', 'log_price', 'city']).drop_nulls().sample(n=5000, seed=SEED)
ac = samp['amenity_count'].to_numpy()
lp = samp['log_price'].to_numpy()
r, p = stats.pearsonr(ac, lp)

fig, ax = plt.subplots(figsize=(9, 5))
for city, label in CITY_LABELS.items():
    sub = samp.filter(pl.col('city') == city)
    ax.scatter(sub['amenity_count'].to_numpy(), sub['log_price'].to_numpy(),
               alpha=0.3, s=15, color=CITY_COLORS[city], label=label)
m, b = np.polyfit(ac, lp, 1)
x_range = np.linspace(ac.min(), ac.max(), 200)
ax.plot(x_range, m * x_range + b, color='black', linewidth=2, label=f'OLS (r = {r:.3f})')
ax.set_title('Amenity Count vs Log-Price (n=5,000 sample)', fontsize=13, fontweight='bold')
ax.set_xlabel('amenity_count'); ax.set_ylabel('log_price')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, '1f_amenity_vs_logprice')
plt.show()
print(f"Pearson r = {r:.4f}  |  p-value = {p:.2e}")

---
## Section 2: Feature Distributions & Correlations

### 2a. Numeric feature summary table

In [ ]:
summary_cols = ['price_usd', 'accommodates', 'bedrooms', 'beds', 'bathrooms',
                'amenity_count', 'years_as_host', 'number_of_reviews',
                'availability_365', 'minimum_nights'] + WALK_COLS
summary_cols = [c for c in summary_cols if c in df.columns]

print(df.select(summary_cols).describe())

### 2b. Correlation with log_price — horizontal bar chart

In [ ]:
exclude = {'id', 'log_price', 'price_usd', 'latitude', 'longitude', 'host_id',
           'neighbourhood_group_cleansed'}
numeric_cols = [
    c for c in df.columns
    if df[c].dtype in (pl.Int8, pl.Int16, pl.Int32, pl.Int64,
                       pl.Float32, pl.Float64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64)
    and c not in exclude
    and (not c.startswith('amenity_') or c == 'amenity_count')
]

corr_items = []
for col in numeric_cols:
    paired = df.select([col, 'log_price']).drop_nulls()
    if len(paired) < 30:
        continue
    r = paired.select(pl.corr(col, 'log_price')).item()
    if r is not None and not np.isnan(r):
        corr_items.append((col, r))

corr_items.sort(key=lambda x: abs(x[1]), reverse=True)
top_corr = corr_items[:20]
names, vals = zip(*top_corr)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#FF5A5F' if v > 0 else '#00A699' for v in vals[::-1]]
bars = ax.barh(list(names[::-1]), list(vals[::-1]), color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
for bar in bars[-10:]:
    bar.set_edgecolor('black'); bar.set_linewidth(1.5)
ax.set_title('Pearson Correlation with log_price (top 20 features)', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson r')
ax.legend(handles=[Patch(color='#FF5A5F', label='Positive'), Patch(color='#00A699', label='Negative')],
          loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_fig(fig, '2b_correlation_barplot')
plt.show()
print("Top 10 correlates with log_price:")
for name, val in top_corr[:10]:
    print(f"  {name:<40} {val:.4f}")

### 2c. Pairplot (seaborn)

In [ ]:
pair_cols = ['log_price', 'accommodates', 'bedrooms', 'amenity_count', 'review_scores_rating']
pair_cols = [c for c in pair_cols if c in df.columns]

# seaborn requires pandas — minimal local conversion for this plot only
pair_df = (
    df
    .select(pair_cols + ['city'])
    .drop_nulls()
    .sample(n=3000, seed=SEED)
    .with_columns(pl.col('city').replace(CITY_LABELS).alias('City'))
    .drop('city')
    .to_pandas()
)
g = sns.pairplot(
    pair_df, hue='City',
    palette={v: CITY_COLORS[k] for k, v in CITY_LABELS.items()},
    plot_kws={'alpha': 0.3, 's': 10},
    diag_kind='kde', corner=True,
)
g.figure.suptitle('Pairplot: Key Features (n=3,000 sample)', y=1.01, fontsize=13, fontweight='bold')
g.figure.savefig('outputs/eda/2c_pairplot.png', dpi=DPI, bbox_inches='tight')
plt.show()

### 2d. Walk Score analysis

In [ ]:
if HAS_WALKSCORE:
    ws_df = df.select(['city', 'log_price'] + WALK_COLS).drop_nulls(subset=WALK_COLS)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, col in zip(axes, WALK_COLS):
        data_list = [ws_df.filter(pl.col('city') == c)[col].drop_nulls().to_numpy()
                     for c in ['nyc', 'la', 'chi']]
        bp = ax.boxplot(data_list, patch_artist=True,
                        medianprops=dict(color='black', linewidth=2))
        for patch, city in zip(bp['boxes'], ['nyc', 'la', 'chi']):
            patch.set_facecolor(CITY_COLORS[city]); patch.set_alpha(0.7)
        ax.set_xticklabels([CITY_LABELS[c] for c in ['nyc', 'la', 'chi']])
        ax.set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
        ax.set_ylabel('Score'); ax.grid(axis='y', alpha=0.3)
    fig.suptitle('Walk / Transit / Bike Scores by City', fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, '2d_walkscores_by_city')
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 6))
    for city, label in CITY_LABELS.items():
        sub = ws_df.filter(pl.col('city') == city).drop_nulls(subset=['walkscore', 'log_price'])
        if len(sub) < 10:
            continue
        wsc = sub['walkscore'].to_numpy()
        lp  = sub['log_price'].to_numpy()
        ax.scatter(wsc, lp, alpha=0.2, s=12, color=CITY_COLORS[city])
        m, b = np.polyfit(wsc, lp, 1)
        r, _ = stats.pearsonr(wsc, lp)
        xs = np.linspace(wsc.min(), wsc.max(), 100)
        ax.plot(xs, m*xs+b, color=CITY_COLORS[city], linewidth=2.5, label=f'{label} (r={r:.2f})')
    ax.set_title('Walk Score vs Log-Price by City', fontsize=13, fontweight='bold')
    ax.set_xlabel('walkscore'); ax.set_ylabel('log_price')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    save_fig(fig, '2d_walkscore_vs_logprice')
    plt.show()

    print("Correlation of walk scores with log_price, by city:")
    for city, label in CITY_LABELS.items():
        sub = ws_df.filter(pl.col('city') == city)
        row_parts = [f"{label:<15}"]
        for col in WALK_COLS:
            paired = sub.select([col, 'log_price']).drop_nulls()
            r = paired.select(pl.corr(col, 'log_price')).item() if len(paired) > 10 else float('nan')
            row_parts.append(f"  {col}: {r:+.3f}")
        print(''.join(row_parts))
else:
    print("Walk score data not available — skipping 2d.")

### 2e. Superhost premium

In [ ]:
sh_agg = (
    df
    .group_by(['city', 'host_is_superhost'])
    .agg(pl.col('price_usd').mean().alias('mean_price'))
    .sort(['city', 'host_is_superhost'])
)

fig, ax = plt.subplots(figsize=(9, 5))
city_order = list(CITY_LABELS.keys())
x = np.arange(len(city_order))
width = 0.35
for i, sh_val in enumerate([0, 1]):
    vals = []
    for c in city_order:
        row = sh_agg.filter((pl.col('city') == c) & (pl.col('host_is_superhost') == sh_val))
        vals.append(row['mean_price'].item() if len(row) > 0 else 0)
    bars = ax.bar(x + i*width - width/2, vals, width,
                  label='Superhost' if sh_val == 1 else 'Non-superhost',
                  color=['#FC642D', '#00A699'][i], alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'${v:.0f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([CITY_LABELS[c] for c in city_order])
ax.set_title('Mean Price: Superhost vs Non-Superhost by City', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean Price (USD)'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_fig(fig, '2e_superhost_premium')
plt.show()

### 2f. Review score vs price

In [ ]:
room_palette = {
    'Entire home/apt': '#FF5A5F', 'Private room': '#00A699',
    'Hotel room': '#FC642D',      'Shared room':  '#767676',
}
rev_samp = (
    df
    .select(['review_scores_rating', 'log_price', 'room_type'])
    .drop_nulls()
    .sample(n=5000, seed=SEED)
)

fig, ax = plt.subplots(figsize=(10, 5))
for rtype, color in room_palette.items():
    sub = rev_samp.filter(pl.col('room_type') == rtype)
    if len(sub) == 0:
        continue
    ax.scatter(sub['review_scores_rating'].to_numpy(), sub['log_price'].to_numpy(),
               alpha=0.3, s=12, color=color, label=rtype)

r, p = stats.pearsonr(rev_samp['review_scores_rating'].to_numpy(), rev_samp['log_price'].to_numpy())
ax.set_title(f'Review Score vs Log-Price by Room Type (r = {r:.3f})', fontsize=13, fontweight='bold')
ax.set_xlabel('review_scores_rating'); ax.set_ylabel('log_price')
ax.legend(title='Room Type', markerscale=2); ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, '2f_review_vs_logprice')
plt.show()
print(f"Pearson r = {r:.4f} | p = {p:.4f}")

---
## Section 3: Geospatial Visualizations

In [ ]:
CITY_CENTERS = {
    'nyc': (40.7128, -74.0060, 11),
    'la':  (34.0522, -118.2437, 10),
    'chi': (41.8781, -87.6298, 11),
}

### 3a. Density heatmaps per city

In [ ]:
for city, label in CITY_LABELS.items():
    sub = df.filter(pl.col('city') == city).drop_nulls(subset=['latitude', 'longitude', 'price_usd'])
    lat_c, lon_c, zoom = CITY_CENTERS[city]
    fig_map = px.density_mapbox(
        sub, lat='latitude', lon='longitude', z='price_usd',
        radius=10, center=dict(lat=lat_c, lon=lon_c), zoom=zoom,
        mapbox_style='carto-positron', color_continuous_scale='RdYlGn_r',
        title=f'{label} — Listing Density & Median Price',
        opacity=0.6, template=PLOTLY_TPL, height=550,
    )
    fig_map.update_layout(coloraxis_colorbar_title='Price (USD)')
    out = f'outputs/maps/{city}_density_heatmap.html'
    fig_map.write_html(out)
    print(f"Saved {out}")
    fig_map.show()

### 3b. Scatter mapbox — 2,000 listings per city colored by log_price

In [ ]:
for city, label in CITY_LABELS.items():
    sub = (
        df
        .filter(pl.col('city') == city)
        .drop_nulls(subset=['latitude', 'longitude', 'log_price'])
        .sample(n=min(2000, df.filter(pl.col('city') == city).height), seed=SEED)
    )
    lat_c, lon_c, zoom = CITY_CENTERS[city]
    fig_s = px.scatter_mapbox(
        sub, lat='latitude', lon='longitude', color='log_price',
        color_continuous_scale='Viridis', size_max=8,
        zoom=zoom, center=dict(lat=lat_c, lon=lon_c),
        mapbox_style='carto-positron',
        title=f'{label} — Sample Listings by Log-Price',
        hover_data=['price_usd', 'room_type', 'neighbourhood_cleansed'],
        opacity=0.75, template=PLOTLY_TPL, height=550,
    )
    fig_s.update_layout(coloraxis_colorbar_title='log_price')
    out = f'outputs/maps/{city}_scatter_logprice.html'
    fig_s.write_html(out)
    print(f"Saved {out}")
    fig_s.show()

### 3c. Neighbourhood price ranking — top 15 & bottom 15 per city (Plotly)

In [ ]:
for city, label in CITY_LABELS.items():
    sub = df.filter(pl.col('city') == city).drop_nulls(subset=['neighbourhood_cleansed', 'price_usd'])

    nbhd_stats = (
        sub
        .group_by('neighbourhood_cleansed')
        .agg([
            pl.col('price_usd').median().alias('median_price'),
            pl.len().alias('n'),
        ])
        .filter(pl.col('n') >= 10)
        .sort('median_price', descending=True)
    )

    top15 = nbhd_stats.head(15).with_columns(pl.lit('Top 15').alias('tier'))
    bot15 = nbhd_stats.tail(15).sort('median_price').with_columns(pl.lit('Bottom 15').alias('tier'))
    combined = pl.concat([bot15, top15])

    fig_n = px.bar(
        combined, x='median_price', y='neighbourhood_cleansed',
        color='tier',
        color_discrete_map={'Top 15': CITY_COLORS[city], 'Bottom 15': '#BBBBBB'},
        orientation='h',
        title=f'{label} — Top 15 & Bottom 15 Neighbourhoods by Median Price',
        labels={'median_price': 'Median Price (USD)', 'neighbourhood_cleansed': ''},
        template=PLOTLY_TPL, height=600,
    )
    fig_n.update_layout(legend_title_text='Tier')
    out = f'outputs/eda/{city}_neighbourhood_ranking.html'
    fig_n.write_html(out)
    print(f"Saved {out}")
    fig_n.show()

---
## Section 4: Hypothesis Testing

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (a.mean() - b.mean()) / pooled_std if pooled_std > 0 else 0

### 4a. Test 1 — Superhost Price Premium

In [ ]:
print(f"{'Group':<12} {'n_super':>8} {'n_non':>8} {'mean_diff':>10} {'t_stat':>9} {'p (1-tail)':>12} {'Cohen d':>9} {'Sig?':>6}")
print('-' * 80)

groups = [('Overall', df)] + [(CITY_LABELS[c], df.filter(pl.col('city') == c)) for c in ['nyc', 'la', 'chi']]

for name, subset in groups:
    sup = subset.filter(pl.col('host_is_superhost') == 1)['log_price'].drop_nulls().to_numpy()
    non = subset.filter(pl.col('host_is_superhost') == 0)['log_price'].drop_nulls().to_numpy()
    t, p2 = stats.ttest_ind(sup, non, equal_var=False)
    p1 = p2 / 2 if t > 0 else 1 - p2 / 2
    d  = cohens_d(sup, non)
    sig = 'Yes' if p1 < 0.05 else 'No'
    print(f"{name:<12} {len(sup):>8} {len(non):>8} {sup.mean()-non.mean():>10.4f} {t:>9.3f} {p1:>12.4e} {d:>9.4f} {sig:>6}")

### 4b. Test 2 — Walkability and Price Tiers (ANOVA + Tukey HSD)

In [ ]:
if HAS_WALKSCORE:
    from statsmodels.stats.multicomp import pairwise_tukeyhsd

    ws_test = df.select(['log_price', 'walkscore']).drop_nulls()
    q33 = ws_test['log_price'].quantile(1/3)
    q66 = ws_test['log_price'].quantile(2/3)

    ws_test = ws_test.with_columns(
        pl.when(pl.col('log_price') <= q33).then(pl.lit('Low'))
          .when(pl.col('log_price') <= q66).then(pl.lit('Mid'))
          .otherwise(pl.lit('High'))
          .alias('tier')
    )

    low  = ws_test.filter(pl.col('tier') == 'Low')['walkscore'].to_numpy()
    mid  = ws_test.filter(pl.col('tier') == 'Mid')['walkscore'].to_numpy()
    high = ws_test.filter(pl.col('tier') == 'High')['walkscore'].to_numpy()

    F, p_anova = stats.f_oneway(low, mid, high)
    print(f"Test 2: One-Way ANOVA — walkscore across price tiers")
    print(f"  F = {F:.4f}  |  p = {p_anova:.4e}")
    print(f"  Tier means: Low={low.mean():.2f}, Mid={mid.mean():.2f}, High={high.mean():.2f}")

    tukey = pairwise_tukeyhsd(
        endog=ws_test['walkscore'].to_numpy(),
        groups=ws_test['tier'].to_numpy(),
        alpha=0.05,
    )
    print("\nTukey HSD post-hoc:")
    print(tukey.summary())
    n_total = len(ws_test)
    eta_sq = (F * 2) / (F * 2 + n_total - 3)
    print(f"  Approx. eta² : {eta_sq:.4f}")

### 4c. Test 3 — Room Type Price Differences Across Cities (ANOVA + Tukey HSD)

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

entire = df.filter(pl.col('room_type') == 'Entire home/apt')
nyc_e = entire.filter(pl.col('city') == 'nyc')['log_price'].drop_nulls().to_numpy()
la_e  = entire.filter(pl.col('city') == 'la')['log_price'].drop_nulls().to_numpy()
chi_e = entire.filter(pl.col('city') == 'chi')['log_price'].drop_nulls().to_numpy()

F3, p3 = stats.f_oneway(nyc_e, la_e, chi_e)
print("Test 3: One-Way ANOVA — log_price of Entire home/apt across cities")
print(f"  F = {F3:.4f}  |  p = {p3:.4e}")
print(f"  Means: NYC={nyc_e.mean():.3f}, LA={la_e.mean():.3f}, Chicago={chi_e.mean():.3f}")

if p3 < 0.05:
    entire_sub = entire.select(['city', 'log_price']).drop_nulls()
    tukey3 = pairwise_tukeyhsd(
        endog=entire_sub['log_price'].to_numpy(),
        groups=entire_sub['city'].to_numpy(),
        alpha=0.05,
    )
    print("\nTukey HSD post-hoc:")
    print(tukey3.summary())
    n3 = len(nyc_e) + len(la_e) + len(chi_e)
    eta_sq3 = (F3 * 2) / (F3 * 2 + n3 - 3)
    print(f"  Approx. eta² : {eta_sq3:.4f}")

---
## Section 5: Feature Engineering Preview

### 5a. Bedrooms × Walk Score interaction

In [ ]:
if HAS_WALKSCORE:
    ws_int = (
        df
        .select(['bedrooms', 'walkscore', 'price_usd'])
        .drop_nulls()
        .filter(pl.col('bedrooms').is_between(1, 4))
        .with_columns(pl.col('bedrooms').cast(pl.Int32))
    )
    ws_arr = ws_int['walkscore'].to_numpy()
    q25, q50, q75 = np.percentile(ws_arr, [25, 50, 75])

    ws_int = ws_int.with_columns(
        pl.when(pl.col('walkscore') <= q25).then(pl.lit('Q1\n(least walkable)'))
          .when(pl.col('walkscore') <= q50).then(pl.lit('Q2'))
          .when(pl.col('walkscore') <= q75).then(pl.lit('Q3'))
          .otherwise(pl.lit('Q4\n(most walkable)'))
          .alias('ws_quartile')
    )

    pivot = (
        ws_int
        .group_by(['ws_quartile', 'bedrooms'])
        .agg(pl.col('price_usd').median().alias('median_price'))
        .sort(['ws_quartile', 'bedrooms'])
    )

    quartile_order = ['Q1\n(least walkable)', 'Q2', 'Q3', 'Q4\n(most walkable)']
    bed_vals = sorted(ws_int['bedrooms'].unique().to_list())
    heat_data = np.full((len(quartile_order), len(bed_vals)), np.nan)
    for row in pivot.iter_rows(named=True):
        qi = quartile_order.index(row['ws_quartile'])
        bi = bed_vals.index(row['bedrooms'])
        heat_data[qi, bi] = row['median_price']

    import pandas as _pd
    heat_df = _pd.DataFrame(heat_data, index=quartile_order, columns=bed_vals)
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.heatmap(heat_df, annot=True, fmt='.0f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'Median Price (USD)'})
    ax.set_title('Median Price: Bedrooms × Walk Score Quartile Interaction',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Bedrooms'); ax.set_ylabel('Walk Score Quartile')
    plt.tight_layout()
    save_fig(fig, '5a_bedroom_walkscore_interaction')
    plt.show()

### 5b. Superhost × Review Score interaction

In [ ]:
int_df = (
    df
    .select(['review_scores_rating', 'log_price', 'host_is_superhost'])
    .drop_nulls()
    .sample(n=min(6000, df.height), seed=SEED)
)

fig, ax = plt.subplots(figsize=(10, 5))
for sh_val, label, color in [(1, 'Superhost', '#FF5A5F'), (0, 'Non-superhost', '#00A699')]:
    sub = int_df.filter(pl.col('host_is_superhost') == sh_val)
    rv  = sub['review_scores_rating'].to_numpy()
    lp  = sub['log_price'].to_numpy()
    ax.scatter(rv, lp, alpha=0.25, s=14, color=color)
    m, b = np.polyfit(rv, lp, 1)
    r, _ = stats.pearsonr(rv, lp)
    xs = np.linspace(rv.min(), rv.max(), 100)
    ax.plot(xs, m*xs+b, color=color, linewidth=2.5, label=f'{label} (r={r:.2f})')

ax.set_title('Superhost × Review Score Interaction on Log-Price', fontsize=13, fontweight='bold')
ax.set_xlabel('review_scores_rating'); ax.set_ylabel('log_price')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, '5b_superhost_review_interaction')
plt.show()

### 5c. Top 20 amenities by correlation with log_price

In [ ]:
amenity_cols = [c for c in df.columns if c.startswith('amenity_') and c != 'amenity_count']

amenity_corrs = []
for col in amenity_cols:
    paired = df.select([col, 'log_price']).drop_nulls()
    if len(paired) < 30:
        continue
    r = paired.select(pl.corr(col, 'log_price')).item()
    if r is not None and not np.isnan(r):
        amenity_corrs.append((col, r))

amenity_corrs.sort(key=lambda x: abs(x[1]), reverse=True)
top20 = amenity_corrs[:20]
names_a, vals_a = zip(*top20)

fig, ax = plt.subplots(figsize=(10, 7))
colors_a = ['#FF5A5F' if v > 0 else '#767676' for v in vals_a[::-1]]
ax.barh(
    [c.replace('amenity_', '').replace('_', ' ').title() for c in names_a[::-1]],
    list(vals_a[::-1]), color=colors_a, edgecolor='white'
)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 20 Amenity Dummies by |Correlation| with log_price', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson r with log_price'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_fig(fig, '5c_amenity_correlations')
plt.show()
print("Top 10 amenities by |r|:")
for name, val in top20[:10]:
    print(f"  {name:<55} {val:+.4f}")